# Green NLP Benchmark Lab — Notebook 01e: Translation and its evaluation — coding workshop (advanced)

**Session 1 of 3, after Notebook 01a.** Part 1 translates one Galician passage four ways,
following the history of machine translation: rules, statistics, a dedicated neural model,
and a general-purpose LLM. Part 2 grades the translations three ways, and measures what the
grading itself costs.

**Before you start:** `Runtime` → `Change runtime type` → select **T4 GPU**, then `Save`.

**At the end of this notebook** you will save `mt_leaderboard.csv` and
`mt_eval_leaderboard.csv`. Download them from the Files sidebar (folder icon, left side),
next to the `leaderboard.csv` from Notebook 01a. Sessions 2 and 3 add rows to all three.

Expected time: 40–60 minutes.

## How this workshop works

This is the coding version of the notebook, **advanced level**. In some code cells, lines
are replaced by `# TODO` comments and `___`. Each exercise above such a cell says what the
code must do; the comments in the cell give the details. This version takes longer than
the expected time given above.

1. Replace every `___` with your code, then run the cell. Open the hints if you need them.
2. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
3. If you are stuck, the solution is below the check cell. Open it after you have tried, or run the
   cell under it: that puts the solution into a new code cell for you, ready to run.
   Or click **Fill in solution** right above the exercise cell, then run the exercise cell:
   the solution replaces the code in it and runs. **Fill in solutions**, under these
   instructions, does this for every exercise.

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

Exercises marked **(advanced)** are only in this version. In them you implement an
equation from the lesson or a paper, or a technique that makes the pipeline cheaper to
run, and check it against the library or the baseline.

In [ ]:
# @title Run this cell first: it sets up the "Fill in solution" buttons (point 3 above).
import re

import ipywidgets as widgets
from IPython.display import display

_solutions = {
    1: "!pip install -q nltk\nfrom nltk.translate import AlignedSent, IBMModel1\nimport re\n\nBITEXT_PAIRS = [\n    (\"ás veces\", \"sometimes\"),\n    (\"é mellor estar calado\", \"it is better to stay quiet\"),\n    (\"unha tía miña\", \"an aunt of mine\"),\n    (\"a miña tía\", \"my aunt\"),\n    (\"todo\", \"everything\"),\n    (\"solucionaba todo\", \"solved everything\"),\n    (\"con ditos e refráns\", \"with sayings and proverbs\"),\n    (\"dicía algo\", \"used to say something\"),\n    (\"dicía algo así\", \"used to say something like this\"),\n    (\"un é escravo\", \"one is a slave\"),\n    (\"escravo do que di\", \"slave of what one says\"),\n    (\"dono do que cala\", \"master of what one keeps quiet\"),\n    (\"sempre me pareceu\", \"it always seemed to me\"),\n    (\"un sabio aforismo\", \"a wise saying\"),\n    (\"se cadra debería\", \"maybe i should have\"),\n    (\"debería terlle feito caso\", \"should have paid heed to it\"),\n    (\"nalgunhas ocasións\", \"on some occasions\"),\n    (\"aquelas ocasións raras\", \"those rare occasions\"),\n    (\"aínda que raras\", \"even though rare\"),\n    (\"contribuíron a dar\", \"contributed to giving\"),\n    (\"bruscos cambios\", \"sudden changes\"),\n    (\"cambios de sentido\", \"changes of meaning\"),\n    (\"á miña vida\", \"to my life\"),\n    (\"a miña vida\", \"my life\"),\n]\n\ntracker_stat = EmissionsTracker(\n    project_name=\"green-mt-statistical\",\n    measure_power_secs=1,\n    save_to_file=False,\n    log_level=\"warning\",\n)\ntracker_stat.start()\nstart_time = time.time()\n\nbitext = [AlignedSent(en.split(), gl.split()) for gl, en in BITEXT_PAIRS]\nibm1_model = IBMModel1(bitext, 20)\n\n\ndef best_translation(src_word, table, min_prob=0.05):\n    candidates = {tgt: table[tgt].get(src_word, 0.0) for tgt in table}\n    if not candidates:\n        return None\n    tgt, prob = max(candidates.items(), key=lambda kv: kv[1])\n    return tgt if prob >= min_prob else None\n\n\ndef translate_statistical(text, table):\n    words = re.findall(r\"[a-záéíóúñ]+\", text.lower())\n    out = [best_translation(w, table) or f\"*{w}\" for w in words]\n    return \" \".join(out)\n\n\ntranslation_stat = translate_statistical(FULL_TEXT_GL, ibm1_model.translation_table)\nelapsed_stat = time.time() - start_time\nemissions_stat_kg = tracker_stat.stop()\n\nprint(translation_stat)\nprint(f\"\\n({elapsed_stat:.4f}s, {emissions_stat_kg * 1000:.6f} g CO2eq)\")",
    2: "!pip install -q sacrebleu\nimport sacrebleu\n\ntracker_bleu = EmissionsTracker(\n    project_name=\"green-eval-bleu\",\n    measure_power_secs=1,\n    save_to_file=False,\n    log_level=\"warning\",\n)\ntracker_bleu.start()\nstart_time = time.time()\n\nbleu_scores = {\n    name: sacrebleu.corpus_bleu([text], [[REFERENCE_EN]]).score\n    for name, text in candidates.items()\n}\n\nelapsed_bleu = time.time() - start_time\nemissions_bleu_kg = tracker_bleu.stop()\n\nfor name, score in bleu_scores.items():\n    print(f\"{name}: BLEU = {score:.2f}\")\nprint(f\"\\n({elapsed_bleu:.4f}s, {emissions_bleu_kg * 1000:.6f} g CO2eq for all candidates)\")",
    3: "import math\nfrom collections import Counter\nfrom sacrebleu.tokenizers.tokenizer_13a import Tokenizer13a\n\ntokenize_13a = Tokenizer13a()\n\n\ndef ngrams(tokens, n):\n    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))\n\n\ndef precision_counts(cand_tokens, ref_tokens, n):\n    cand, ref = ngrams(cand_tokens, n), ngrams(ref_tokens, n)\n    correct = sum(min(count, ref[g]) for g, count in cand.items())\n    total = sum(cand.values())\n    return correct, total\n\n\ndef smoothed_precisions(correct, total):\n    \"\"\"Given: sacrebleu's default smoothing. Each zero count gets 1 / (2^k * total).\"\"\"\n    precisions, factor = [], 1.0\n    for c, t in zip(correct, total):\n        if c == 0:\n            factor *= 2\n            precisions.append(1 / (factor * t))\n        else:\n            precisions.append(c / t)\n    return precisions\n\n\ndef my_bleu(candidate, reference, max_n=4):\n    cand_tokens = tokenize_13a(candidate).split()\n    ref_tokens = tokenize_13a(reference).split()\n    counts = [precision_counts(cand_tokens, ref_tokens, n) for n in range(1, max_n + 1)]\n    correct = [c for c, _ in counts]\n    total = [t for _, t in counts]\n    if sum(correct) == 0:\n        return 0.0\n    c, r = len(cand_tokens), len(ref_tokens)\n    bp = 1.0 if c >= r else math.exp(1 - r / c)\n    precisions = smoothed_precisions(correct, total)\n    return 100 * bp * math.exp(sum(math.log(p) for p in precisions) / max_n)\n\n\nfor name, text in candidates.items():\n    print(f\"{name:18s} mine = {my_bleu(text, REFERENCE_EN):6.2f}   sacrebleu = {bleu_scores[name]:6.2f}\")",
    4: "import re\n\n\ndef parse_score(value):\n    try:\n        return float(value)\n    except (TypeError, ValueError):\n        match = re.search(r\"-?\\d+(\\.\\d+)?\", str(value))\n        return float(match.group()) if match else None\n\n\nscore_sets = [(\"BLEU\", bleu_scores)]\nif bertscore_available:\n    score_sets.append((\"BERTScore\", bertscore_scores))\nscore_sets.append((\"LLM-judge (Qwen)\", {k: parse_score(v) for k, v in judge_scores.items()}))\nscore_sets.append((\"LLM-judge (ChatGPT)\", {k: parse_score(v) for k, v in chatgpt_judge_scores.items()}))\n\nfig, axes = plt.subplots(1, len(score_sets), figsize=(4.5 * len(score_sets), 4))\nif len(score_sets) == 1:\n    axes = [axes]\n\nfor ax, (metric_name, scores) in zip(axes, score_sets):\n    names = list(scores.keys())\n    values = [scores[n] for n in names]\n    ax.bar(names, values)\n    ax.set_title(metric_name)\n    ax.tick_params(axis=\"x\", rotation=30)\n\nplt.tight_layout()\nplt.show()",
}
_fill_in = globals().get("_fill_in", set())


def _fill_in_transformer(lines):
    """An exercise cell starts with `# Exercise N`; with fill-in on for N, it becomes the solution."""
    m = re.match(r"# Exercise (\d+)\s*$", lines[0]) if lines else None
    if not m or int(m.group(1)) not in _fill_in:
        return lines
    n = int(m.group(1))
    return [f"get_ipython().set_next_input(_solutions[{n}], replace=True)\n"] + (_solutions[n] + "\n").splitlines(True)


_ip = get_ipython()
_ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup
                                     if getattr(t, "__name__", "") != "_fill_in_transformer"]
_ip.input_transformers_cleanup.append(_fill_in_transformer)


def _fill_in_button(exercises, label, message):
    out = widgets.Output()

    def click(_):
        _fill_in.update(exercises)
        with out:
            out.clear_output()
            print(message)

    button = widgets.Button(description=label, icon="magic", layout=widgets.Layout(width="auto"))
    button.on_click(click)
    display(widgets.HBox([button, out]))


_fill_in_button(_solutions, "Fill in solutions",
                "✓ Every exercise cell you run from now on gets its solution.")

## Step 0 — Install, import, and load the general-purpose model

The same model as Notebook 01a, Qwen2.5-1.5B-Instruct. It translates in Step 6 and grades
in Step 12.

In [ ]:
!pip install -q transformers accelerate codecarbon pandas

In [ ]:
import time
import textwrap
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from codecarbon import EmissionsTracker

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — the notebook will still run, just slower. "
          "Double check Runtime > Change runtime type > T4 GPU.")

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
print(f"Loaded {MODEL_NAME}")
print(f"Parameter count: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## Part 1 — Machine translation: four eras, one sentence

The sentence:

> *"Ás veces é mellor estar calado. Unha tía miña, que todo o solucionaba con ditos e
> refráns, dicía algo así como 'un é escravo do que di e dono do que cala'."*

It is Galician — a low-resource language — from the novel
*Speechwriter* by Paulo Naseiro. In the lesson, the fragment "unha tía miña" was aligned by hand
with attention.

Now you are going to translate it with **four different kinds of system**, tracing that
same historical arc:

1. **Rule-based** (1970s-2000s style) — [Apertium](https://apertium.org), a real,
   actively maintained rule-based MT engine, called through its public API
2. **Statistical** (1990s style) — IBM Model 1, which learns word translations from
   example sentence pairs, then translates word by word
3. **Neural, dedicated** (2015-style architecture) — a small model trained for exactly one
   job: Galician → English
4. **General-purpose LLM** (today) — the same model you just used for question answering,
   simply asked to translate instead

Approach 2 is a real algorithm on a toy training set: 24 short phrase pairs, written for
this notebook. Approaches 1, 3, and 4 are real systems. All four translate the whole
passage.

*Want to use the first chapter of* Speechwriter*, or a different Galician text entirely, for
approaches 1, 3, and 4?* Ask your instructor for the chapter text, or paste in any Galician
text you like in Step 2 — or flip the direction and translate English **into** Galician
(see the stretch goal at the end of Part 1).

## Step 1 — Install the extra piece we need, and a shared helper

The general-purpose model is already loaded, in Step 0. The dedicated translation model
uses a different tokenizer library (`sentencepiece`), so we install that here. We also
define one small helper function, reused for every measurement in Parts 1 and 2, that pulls
the total energy out of a `CodeCarbon` tracker the same way Notebook 01a does.

In [ ]:
!pip install -q sentencepiece sacremoses

from transformers import AutoModelForSeq2SeqLM


def get_tracker_energy_kwh(tracker):
    for attr_path in ("final_emissions_data.energy_consumed", "_total_energy.kWh"):
        obj = tracker
        try:
            for part in attr_path.split("."):
                obj = getattr(obj, part)
            return float(obj)
        except Exception:
            continue
    return None

## Step 2 — The text

`FULL_TEXT_GL` is the whole passage. All four translation systems below work on it
directly. Replace it with a longer passage or a different Galician text if you like — see
the stretch goal at the end of Part 1.

In [ ]:
FULL_TEXT_GL = (
    "Ás veces é mellor estar calado. Unha tía miña, que todo o solucionaba con ditos e "
    "refráns, dicía algo así como \"un é escravo do que di e dono do que cala\". Sempre me "
    "pareceu un sabio aforismo, e se cadra debería terlle feito caso nalgunhas ocasións, "
    "aquelas que, aínda que raras, contribuíron a dar bruscos cambios de sentido á miña vida."
)

print(f"Full text ({len(FULL_TEXT_GL.split())} words):\n{FULL_TEXT_GL}")

## Step 3 — Approach 1: rule-based, for real (1970s-2000s style)

A real system this time. [Apertium](https://apertium.org) is an open-source, actively
maintained rule-based MT engine: morphological analysis, part-of-speech disambiguation,
structural transfer rules, then generation — no statistics, no neural network anywhere in
the pipeline. It has a real Galician→English pair, and we call its public translation API
directly.

**Known limitation, worth noticing:** this translation happens on Apertium's own server.
The CodeCarbon numbers below measure *our local network wait*, standing in for a
server-side compute cost that stays invisible to us — genuinely hard to cost out
honestly, worth flagging rather than glossing over.

In [ ]:
import requests

APERTIUM_URL = "https://apertium.org/apy/translate"
APERTIUM_FALLBACK = (
    "A mine aunt, than everything solved it with said and sayings, said something as "
    "well as one is enslaved of the that says and owner of the that cove."
)


def translate_rule_based(text, langpair="glg|eng"):
    response = requests.get(APERTIUM_URL, params={"langpair": langpair, "q": text}, timeout=15)
    response.raise_for_status()
    return response.json()["responseData"]["translatedText"]


tracker_rule = EmissionsTracker(
    project_name="green-mt-rule-based",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_rule.start()
start_time = time.time()
try:
    translation_rule = translate_rule_based(FULL_TEXT_GL)
except Exception as e:
    print(f"Apertium API call failed ({e}) — using a real response captured earlier.")
    translation_rule = APERTIUM_FALLBACK
elapsed_rule = time.time() - start_time
emissions_rule_kg = tracker_rule.stop()

print(f"\"{FULL_TEXT_GL}\"")
print(f"-> \"{translation_rule}\"")
print(f"({elapsed_rule:.4f}s, {emissions_rule_kg * 1000:.6f} g CO2eq measured locally — the")
print("actual translation ran on Apertium's server.)")

### Is this Galician being under-served, or is Apertium just bad?

Neither, it turns out. Below is the exact same engine and the exact same
`translate_rule_based` function, translating two plain, non-idiomatic Spanish sentences —
Spanish is one of Apertium's most mature language pairs, nothing like Galician's smaller
coverage. If the garbled output above were really about Galician being under-resourced,
or Apertium just being a bad tool, plain Spanish sentences should look just as broken.

In [ ]:
SPANISH_PLAIN_SENTENCES = [
    "El gato negro duerme en la silla de la cocina todos los días.",
    "Mi hermana vive en una ciudad grande cerca del mar.",
]

SPANISH_FALLBACKS = [
    "The black cat sleeps in the chair of the cookery every day.",
    "My sister lives in a big city near of the mar.",
]

for sentence, fallback in zip(SPANISH_PLAIN_SENTENCES, SPANISH_FALLBACKS):
    try:
        translation = translate_rule_based(sentence, langpair="spa|eng")
    except Exception as e:
        print(f"Apertium API call failed ({e}) — using a real response captured earlier.")
        translation = fallback
    print(f"\"{sentence}\"")
    print(f"-> \"{translation}\"\n")

print("Correct word order, correct tense, mostly the right words — genuinely usable for")
print("plain declarative text. So the garbled Galician output above isn't Apertium being")
print("broken, and it isn't Galician being under-supported: it's rule-based MT hitting its")
print("real limitation — no representation of meaning, just structural substitution — on a")
print("sentence chosen specifically because it's proverb-heavy and idiomatic.")

**Reflection:** Look closely at the second sentence's output: *"My sister lives in a
big city near of the mar."* Something is genuinely wrong with it, in two separate ways —
one word problem and one grammar problem. What are they? (Try translating "mar" by
itself, and "cerca de" by itself, to investigate.)

## Step 4 — Approach 2: statistical, IBM Model 1 (1990s style)

A real implementation this time, not hand-invented: NLTK's `IBMModel1`, the actual
foundational word-alignment algorithm from the statistical MT research literature (Brown
et al., early 1990s). It is trained here on a small, hand-built Galician-English parallel
corpus --- nowhere near what a real system of that era would have used, but a real
algorithm learning real, if sparse, translation probabilities from real example
sentences, not a hand-written dictionary.

Decoding is deliberately the simplest possible version: pick the most likely English word
for each Galician word, in the original word order, with no language model and no
reordering search. That simplicity is the actual point, not a shortcut --- see the note
after the output below.

### ✏️ Exercise 1 — Pick the most probable translation of a word

In `best_translation`, `candidates` maps each English word to P(English word |
Galician word). Return the English word with the highest probability, or `None` if even
that probability is below `min_prob`.

<details><summary><b>Hints</b> (click to open)</summary>

- `candidates.items()` gives `(word, probability)` pairs; `kv[1]` is the probability.
- Return `None` when `prob` is smaller than `min_prob`.

</details>

In [ ]:
# @title Fill in solution (Exercise 1)
_fill_in_button([1], "Fill in solution",
                "✓ Now run the exercise cell below: the solution replaces its code.")

In [ ]:
# Exercise 1
!pip install -q nltk
from nltk.translate import AlignedSent, IBMModel1
import re

BITEXT_PAIRS = [
    ("ás veces", "sometimes"),
    ("é mellor estar calado", "it is better to stay quiet"),
    ("unha tía miña", "an aunt of mine"),
    ("a miña tía", "my aunt"),
    ("todo", "everything"),
    ("solucionaba todo", "solved everything"),
    ("con ditos e refráns", "with sayings and proverbs"),
    ("dicía algo", "used to say something"),
    ("dicía algo así", "used to say something like this"),
    ("un é escravo", "one is a slave"),
    ("escravo do que di", "slave of what one says"),
    ("dono do que cala", "master of what one keeps quiet"),
    ("sempre me pareceu", "it always seemed to me"),
    ("un sabio aforismo", "a wise saying"),
    ("se cadra debería", "maybe i should have"),
    ("debería terlle feito caso", "should have paid heed to it"),
    ("nalgunhas ocasións", "on some occasions"),
    ("aquelas ocasións raras", "those rare occasions"),
    ("aínda que raras", "even though rare"),
    ("contribuíron a dar", "contributed to giving"),
    ("bruscos cambios", "sudden changes"),
    ("cambios de sentido", "changes of meaning"),
    ("á miña vida", "to my life"),
    ("a miña vida", "my life"),
]

tracker_stat = EmissionsTracker(
    project_name="green-mt-statistical",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_stat.start()
start_time = time.time()

bitext = [AlignedSent(en.split(), gl.split()) for gl, en in BITEXT_PAIRS]
ibm1_model = IBMModel1(bitext, 20)


def best_translation(src_word, table, min_prob=0.05):
    candidates = {tgt: table[tgt].get(src_word, 0.0) for tgt in table}
    if not candidates:
        return None
    # TODO: the most probable target word, or None below min_prob
    ___


def translate_statistical(text, table):
    words = re.findall(r"[a-záéíóúñ]+", text.lower())
    out = [best_translation(w, table) or f"*{w}" for w in words]
    return " ".join(out)


translation_stat = translate_statistical(FULL_TEXT_GL, ibm1_model.translation_table)
elapsed_stat = time.time() - start_time
emissions_stat_kg = tracker_stat.stop()

print(translation_stat)
print(f"\n({elapsed_stat:.4f}s, {emissions_stat_kg * 1000:.6f} g CO2eq)")

In [ ]:
# Check for Exercise 1. Run this cell as is.
_t = {"aunt": {"tía": 0.9}, "my": {"tía": 0.1, "miña": 0.8}}
assert best_translation("tía", _t) == "aunt"
assert best_translation("miña", _t) == "my"
assert best_translation("xyz", _t) is None
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
!pip install -q nltk
from nltk.translate import AlignedSent, IBMModel1
import re

BITEXT_PAIRS = [
    ("ás veces", "sometimes"),
    ("é mellor estar calado", "it is better to stay quiet"),
    ("unha tía miña", "an aunt of mine"),
    ("a miña tía", "my aunt"),
    ("todo", "everything"),
    ("solucionaba todo", "solved everything"),
    ("con ditos e refráns", "with sayings and proverbs"),
    ("dicía algo", "used to say something"),
    ("dicía algo así", "used to say something like this"),
    ("un é escravo", "one is a slave"),
    ("escravo do que di", "slave of what one says"),
    ("dono do que cala", "master of what one keeps quiet"),
    ("sempre me pareceu", "it always seemed to me"),
    ("un sabio aforismo", "a wise saying"),
    ("se cadra debería", "maybe i should have"),
    ("debería terlle feito caso", "should have paid heed to it"),
    ("nalgunhas ocasións", "on some occasions"),
    ("aquelas ocasións raras", "those rare occasions"),
    ("aínda que raras", "even though rare"),
    ("contribuíron a dar", "contributed to giving"),
    ("bruscos cambios", "sudden changes"),
    ("cambios de sentido", "changes of meaning"),
    ("á miña vida", "to my life"),
    ("a miña vida", "my life"),
]

tracker_stat = EmissionsTracker(
    project_name="green-mt-statistical",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_stat.start()
start_time = time.time()

bitext = [AlignedSent(en.split(), gl.split()) for gl, en in BITEXT_PAIRS]
ibm1_model = IBMModel1(bitext, 20)


def best_translation(src_word, table, min_prob=0.05):
    candidates = {tgt: table[tgt].get(src_word, 0.0) for tgt in table}
    if not candidates:
        return None
    tgt, prob = max(candidates.items(), key=lambda kv: kv[1])
    return tgt if prob >= min_prob else None


def translate_statistical(text, table):
    words = re.findall(r"[a-záéíóúñ]+", text.lower())
    out = [best_translation(w, table) or f"*{w}" for w in words]
    return " ".join(out)


translation_stat = translate_statistical(FULL_TEXT_GL, ibm1_model.translation_table)
elapsed_stat = time.time() - start_time
emissions_stat_kg = tracker_stat.stop()

print(translation_stat)
print(f"\n({elapsed_stat:.4f}s, {emissions_stat_kg * 1000:.6f} g CO2eq)")
```

</details>

In [ ]:
# Stuck on Exercise 1? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 1" above; your own cell stays as it is.)
get_ipython().set_next_input("!pip install -q nltk\nfrom nltk.translate import AlignedSent, IBMModel1\nimport re\n\nBITEXT_PAIRS = [\n    (\"ás veces\", \"sometimes\"),\n    (\"é mellor estar calado\", \"it is better to stay quiet\"),\n    (\"unha tía miña\", \"an aunt of mine\"),\n    (\"a miña tía\", \"my aunt\"),\n    (\"todo\", \"everything\"),\n    (\"solucionaba todo\", \"solved everything\"),\n    (\"con ditos e refráns\", \"with sayings and proverbs\"),\n    (\"dicía algo\", \"used to say something\"),\n    (\"dicía algo así\", \"used to say something like this\"),\n    (\"un é escravo\", \"one is a slave\"),\n    (\"escravo do que di\", \"slave of what one says\"),\n    (\"dono do que cala\", \"master of what one keeps quiet\"),\n    (\"sempre me pareceu\", \"it always seemed to me\"),\n    (\"un sabio aforismo\", \"a wise saying\"),\n    (\"se cadra debería\", \"maybe i should have\"),\n    (\"debería terlle feito caso\", \"should have paid heed to it\"),\n    (\"nalgunhas ocasións\", \"on some occasions\"),\n    (\"aquelas ocasións raras\", \"those rare occasions\"),\n    (\"aínda que raras\", \"even though rare\"),\n    (\"contribuíron a dar\", \"contributed to giving\"),\n    (\"bruscos cambios\", \"sudden changes\"),\n    (\"cambios de sentido\", \"changes of meaning\"),\n    (\"á miña vida\", \"to my life\"),\n    (\"a miña vida\", \"my life\"),\n]\n\ntracker_stat = EmissionsTracker(\n    project_name=\"green-mt-statistical\",\n    measure_power_secs=1,\n    save_to_file=False,\n    log_level=\"warning\",\n)\ntracker_stat.start()\nstart_time = time.time()\n\nbitext = [AlignedSent(en.split(), gl.split()) for gl, en in BITEXT_PAIRS]\nibm1_model = IBMModel1(bitext, 20)\n\n\ndef best_translation(src_word, table, min_prob=0.05):\n    candidates = {tgt: table[tgt].get(src_word, 0.0) for tgt in table}\n    if not candidates:\n        return None\n    tgt, prob = max(candidates.items(), key=lambda kv: kv[1])\n    return tgt if prob >= min_prob else None\n\n\ndef translate_statistical(text, table):\n    words = re.findall(r\"[a-záéíóúñ]+\", text.lower())\n    out = [best_translation(w, table) or f\"*{w}\" for w in words]\n    return \" \".join(out)\n\n\ntranslation_stat = translate_statistical(FULL_TEXT_GL, ibm1_model.translation_table)\nelapsed_stat = time.time() - start_time\nemissions_stat_kg = tracker_stat.stop()\n\nprint(translation_stat)\nprint(f\"\\n({elapsed_stat:.4f}s, {emissions_stat_kg * 1000:.6f} g CO2eq)\")")


**Why this looks like word salad, and why that's the point:** repeated words like "stay
stay stay" or "with with with with" are not a bug in the decoding above --- they are a
real, well-documented property of translating word-by-word with no language model and no
reordering: several Galician words end up tied, or nearly tied, across multiple English
words in such a small training corpus, and independently picking the single most likely
English word for each one can pick the same word repeatedly. This is close to what the
earliest purely word-based statistical MT systems actually produced, and it is a real
part of why the field moved toward phrase-based systems and, eventually, systems with an
integrated language model choosing word order. Words marked with a leading `*` never
appeared on the Galician side of the training corpus at all --- genuinely unknown to this
tiny model, the same kind of gap Apertium showed with `*calado` earlier, just for a
different reason (missing training data here, rather than a missing dictionary entry
there).


## Step 5 — Approach 3: a dedicated neural model (2015-style architecture)

`Helsinki-NLP/opus-mt-gl-en` does exactly one job — Galician to English — using the
encoder-decoder-with-attention architecture. It is far smaller
than the general-purpose model below; the first run downloads it (roughly 300MB). This one
gets the **whole passage**.

In [ ]:
MT_MODEL_NAME = "Helsinki-NLP/opus-mt-gl-en"

mt_tokenizer = AutoTokenizer.from_pretrained(MT_MODEL_NAME)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL_NAME).to(model.device)
print(f"Loaded {MT_MODEL_NAME}")
print(f"Parameter count: {sum(p.numel() for p in mt_model.parameters()) / 1e6:.1f}M")


def translate_with_dedicated_model(text):
    inputs = mt_tokenizer(text, return_tensors="pt", truncation=True).to(mt_model.device)
    with torch.no_grad():
        output = mt_model.generate(**inputs, max_new_tokens=200)
    return mt_tokenizer.decode(output[0], skip_special_tokens=True)


tracker_neural = EmissionsTracker(
    project_name="green-mt-neural-dedicated",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_neural.start()
start_time = time.time()
translation_neural = translate_with_dedicated_model(FULL_TEXT_GL)
elapsed_neural = time.time() - start_time
emissions_neural_kg = tracker_neural.stop()

print(f"({elapsed_neural:.2f}s, {emissions_neural_kg * 1000:.4f} g CO2eq)\n")
print(translation_neural)

## Step 6 — Approach 4: the general-purpose LLM

The same model, same kind of prompt as Notebook 01a — just asked to translate instead of answer a
question.

In [ ]:
def translate_with_llm(text, source_lang="Galician", target_lang="English"):
    messages = [
        {
            "role": "system",
            "content": (
                f"You are a professional translator. Translate the user's {source_lang} "
                f"text into fluent {target_lang}. Output only the translation."
            ),
        },
        {"role": "user", "content": text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


tracker_llm = EmissionsTracker(
    project_name="green-mt-general-llm",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_llm.start()
start_time = time.time()
translation_llm = translate_with_llm(FULL_TEXT_GL)
elapsed_llm = time.time() - start_time
emissions_llm_kg = tracker_llm.stop()

print(f"({elapsed_llm:.2f}s, {emissions_llm_kg * 1000:.4f} g CO2eq)\n")
print(translation_llm)

## Step 7 — Compare all four

Same underlying sentence, four very different systems and costs.

In [ ]:
mt_comparison = pd.DataFrame([
    {"era": "1. Rule-based (real)", "model": "Apertium (glg-eng)", "params": "n/a (rule-based)",
     "scope": "full passage", "duration_sec": round(elapsed_rule, 4),
     "emissions_g_co2eq": round(emissions_rule_kg * 1000, 6)},
    {"era": "2. Statistical", "model": "IBM Model 1, 24 phrase pairs", "params": "n/a",
     "scope": "full passage", "duration_sec": round(elapsed_stat, 4),
     "emissions_g_co2eq": round(emissions_stat_kg * 1000, 6)},
    {"era": "3. Neural, dedicated", "model": MT_MODEL_NAME,
     "params": f"{sum(p.numel() for p in mt_model.parameters()) / 1e6:.0f}M",
     "scope": "full passage", "duration_sec": round(elapsed_neural, 2),
     "emissions_g_co2eq": round(emissions_neural_kg * 1000, 4)},
    {"era": "4. General-purpose LLM", "model": MODEL_NAME,
     "params": f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B",
     "scope": "full passage", "duration_sec": round(elapsed_llm, 2),
     "emissions_g_co2eq": round(emissions_llm_kg * 1000, 4)},
])

print(mt_comparison.to_string(index=False))

print("\nLook at the neural and LLM translations again. How does each introduce the aunt?")
print("  'my aunt' assumes the reader already knows her;")
print("  'an aunt of mine' keeps all three Galician words;")
print("  'I had an aunt who...' restructures the sentence, as English books usually do.")
print("Restructuring takes more than better words or a better word order: that is the")
print("alignment-vs-translation gap from the lesson. Try the Galician sentence in a large")
print("commercial model (ChatGPT, Claude, Gemini): which opening does it choose?")

### Seeing the cost spread

In [ ]:
import matplotlib.pyplot as plt

bar_colors = ["tab:orange" if era.startswith("1.") else "tab:blue" for era in mt_comparison["era"]]
cheapest = mt_comparison.loc[mt_comparison["emissions_g_co2eq"].idxmin()]
priciest = mt_comparison.loc[mt_comparison["emissions_g_co2eq"].idxmax()]
ratio = priciest["emissions_g_co2eq"] / max(cheapest["emissions_g_co2eq"], 1e-12)

plt.figure(figsize=(8, 4))
plt.bar(mt_comparison["era"], mt_comparison["emissions_g_co2eq"], color=bar_colors)
plt.yscale("log")
plt.ylabel("g CO2eq (log scale)")
plt.title(f"Priciest vs. cheapest translation approach: ~{ratio:,.0f}x apart")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

print(f"Cheapest: {cheapest['era']} ({cheapest['emissions_g_co2eq']:.6f} g CO2eq).")
print(f"Priciest: {priciest['era']} ({priciest['emissions_g_co2eq']:.4f} g CO2eq).")
print("The orange bar (rule-based) measures local network wait — Apertium's real")
print("server-side compute stays invisible to us here (see Step 3). Read it as a lower")
print("bound, a different kind of measurement from the other three bars.")

## Step 8 — Log this comparison

In [ ]:
mt_leaderboard = pd.DataFrame([
    {"task": "translation", "era": "rule-based", "model": "apertium-glg-eng",
     "duration_sec": elapsed_rule, "energy_kwh": get_tracker_energy_kwh(tracker_rule),
     "emissions_kg_co2eq": emissions_rule_kg},
    {"task": "translation", "era": "statistical", "model": "ibm-model-1",
     "duration_sec": elapsed_stat, "energy_kwh": get_tracker_energy_kwh(tracker_stat),
     "emissions_kg_co2eq": emissions_stat_kg},
    {"task": "translation", "era": "neural-dedicated", "model": MT_MODEL_NAME,
     "duration_sec": elapsed_neural, "energy_kwh": get_tracker_energy_kwh(tracker_neural),
     "emissions_kg_co2eq": emissions_neural_kg},
    {"task": "translation", "era": "general-llm", "model": MODEL_NAME,
     "duration_sec": elapsed_llm, "energy_kwh": get_tracker_energy_kwh(tracker_llm),
     "emissions_kg_co2eq": emissions_llm_kg},
])
mt_leaderboard.to_csv("mt_leaderboard.csv", index=False)
mt_leaderboard

In [ ]:
import os

if os.path.exists("mt_leaderboard.csv"):
    print("Saved mt_leaderboard.csv in this Colab session.")
    print("To download it: click the folder icon in the left sidebar, find")
    print("mt_leaderboard.csv, right-click it, and choose Download.")
else:
    print("mt_leaderboard.csv not found — re-run the cell above first.")

### Optional stretch goal — flip the direction, or change the text

If you have time:

- **Translate English into Galician instead**: Apertium also has a real English→Galician
  pair — pass `langpair="eng|glg"` to `translate_rule_based` in Step 3. For the neural
  model, set `MT_MODEL_NAME = "Helsinki-NLP/opus-mt-en-gl"` and re-run Step 5, and change
  `source_lang`/`target_lang` in Step 6's call to `translate_with_llm` to translate *into*
  Galician instead of out of it.
- **Use a longer or different text**: replace `FULL_TEXT_GL` in Step 2 with a longer
  passage (e.g. the rest of *Speechwriter*'s first chapter, if your instructor has shared
  it) or any other Galician text, and re-run Steps 10 and 13–16. Does the cost gap between
  the neural and general-purpose approaches grow, shrink, or stay about the same as the
  text gets longer? Why, given how attention cost scales with context length?

## Part 2 — Evaluating translations (and what evaluation itself costs)

So which translation was actually *good*? "Compare them yourself" doesn't scale past a
handful of sentences — real MT systems are evaluated automatically, against a reference
translation written by a human. We'll try three evaluation methods, from oldest and
cheapest to newest and most expensive, and measure the cost of **evaluating** too.

**Step 9 — The reference translation.** `REFERENCE_EN` below is one reasonable human translation, incorporating the "I had an aunt
who..." fix. Feel free to write your own instead — the point of this section
holds either way, and a metric is only as good as the reference it's compared against,
which is itself a real cost (a human had to write it).

In [ ]:
REFERENCE_EN = (
    "Sometimes it's better to stay quiet. I had an aunt who solved everything with sayings "
    "and proverbs; she used to say something like, 'You're a slave to what you say, and "
    "master of what you keep to yourself.' It always struck me as a wise saying, and maybe "
    "I should have listened to it on a few occasions — the rare ones that, even so, ended up "
    "sharply changing the course of my life."
)

candidates = {
    "rule-based": translation_rule,
    "statistical": translation_stat,
    "neural-dedicated": translation_neural,
    "general-llm": translation_llm,
}

for name, text in candidates.items():
    print(f"--- {name} ---")
    print(text)
    print()
print("--- reference ---")
print(REFERENCE_EN)

**Scope note:** `statistical` now translates the same `FULL_TEXT_GL` as the other three
systems, word-by-word with no reordering (see Step 4). Its BLEU/BERTScore/judge scores
below will still be low, but for an honest reason now: repeated and unknown (`*`-marked)
words, not a narrower scope than everyone else's. It is still worth scoring: the point is
to see the actual number a monotone, language-model-free decoder produces, not to assume
it.


## Step 10 — BLEU (2002): count overlapping n-grams

BLEU just counts how many word sequences (n-grams) a candidate translation shares with the
reference. No model, no GPU — pure counting, the same idea as n-gram counting.

### ✏️ Exercise 2 — BLEU for every candidate

Compute the BLEU score of each candidate translation against `REFERENCE_EN`.
`sacrebleu.corpus_bleu` takes a list of system outputs and a list of reference lists.

<details><summary><b>Hints</b> (click to open)</summary>

- The candidate goes in the first list, the reference in the nested second list.
- Swapping them gives a different number.

</details>

In [ ]:
# @title Fill in solution (Exercise 2)
_fill_in_button([2], "Fill in solution",
                "✓ Now run the exercise cell below: the solution replaces its code.")

In [ ]:
# Exercise 2
!pip install -q sacrebleu
import sacrebleu

tracker_bleu = EmissionsTracker(
    project_name="green-eval-bleu",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_bleu.start()
start_time = time.time()

bleu_scores = ___  # TODO: {candidate name: BLEU score against REFERENCE_EN}

elapsed_bleu = time.time() - start_time
emissions_bleu_kg = tracker_bleu.stop()

for name, score in bleu_scores.items():
    print(f"{name}: BLEU = {score:.2f}")
print(f"\n({elapsed_bleu:.4f}s, {emissions_bleu_kg * 1000:.6f} g CO2eq for all candidates)")

In [ ]:
# Check for Exercise 2. Run this cell as is.
assert set(bleu_scores) == set(candidates)
assert all(0 <= s <= 100 for s in bleu_scores.values())
_n = "general-llm"
assert abs(bleu_scores[_n] - sacrebleu.corpus_bleu([candidates[_n]], [[REFERENCE_EN]]).score) < 1e-9, \
    "the candidate goes first, the reference second"
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
!pip install -q sacrebleu
import sacrebleu

tracker_bleu = EmissionsTracker(
    project_name="green-eval-bleu",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_bleu.start()
start_time = time.time()

bleu_scores = {
    name: sacrebleu.corpus_bleu([text], [[REFERENCE_EN]]).score
    for name, text in candidates.items()
}

elapsed_bleu = time.time() - start_time
emissions_bleu_kg = tracker_bleu.stop()

for name, score in bleu_scores.items():
    print(f"{name}: BLEU = {score:.2f}")
print(f"\n({elapsed_bleu:.4f}s, {emissions_bleu_kg * 1000:.6f} g CO2eq for all candidates)")
```

</details>

In [ ]:
# Stuck on Exercise 2? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 2" above; your own cell stays as it is.)
get_ipython().set_next_input("!pip install -q sacrebleu\nimport sacrebleu\n\ntracker_bleu = EmissionsTracker(\n    project_name=\"green-eval-bleu\",\n    measure_power_secs=1,\n    save_to_file=False,\n    log_level=\"warning\",\n)\ntracker_bleu.start()\nstart_time = time.time()\n\nbleu_scores = {\n    name: sacrebleu.corpus_bleu([text], [[REFERENCE_EN]]).score\n    for name, text in candidates.items()\n}\n\nelapsed_bleu = time.time() - start_time\nemissions_bleu_kg = tracker_bleu.stop()\n\nfor name, score in bleu_scores.items():\n    print(f\"{name}: BLEU = {score:.2f}\")\nprint(f\"\\n({elapsed_bleu:.4f}s, {emissions_bleu_kg * 1000:.6f} g CO2eq for all candidates)\")")


### ✏️ Exercise 3 — BLEU from its definition (advanced)

`sacrebleu` computed BLEU in one line. Here you compute it from its definition
(Papineni et al. 2002) and check that you get the same numbers.

1. **Clipped n-gram precision** p_n, for n = 1 … 4. Each candidate n-gram counts as correct
   at most as many times as it occurs in the reference:
   p_n = (correct n-grams) / (all n-grams in the candidate).
2. **Brevity penalty.** With candidate length c and reference length r, in tokens:
   BP = 1 if c ≥ r, and BP = exp(1 − r / c) if c < r.
3. **BLEU** = 100 · BP · exp( ¼ · (log p_1 + log p_2 + log p_3 + log p_4) ),
   that is, 100 · BP · the geometric mean of the four precisions.

If some p_n is 0, its log is −∞ and BLEU is 0. By default, sacrebleu replaces each zero
precision with a small number instead; the given function `smoothed_precisions` does this
the same way. Both texts are split into tokens with sacrebleu's own tokenizer, so that the
counts match.

<details><summary><b>Hints</b> (click to open)</summary>

- `tuple(tokens[i:i + n])` is the n-gram starting at position i; there are `len(tokens) - n + 1` of them.
- Clipping: `sum(min(count, ref[g]) for g, count in cand.items())`. A Counter returns 0 for a missing key.
- `math.exp(1 - r / c)` when `c < r`, else 1.
- The geometric mean of the precisions is `math.exp(sum(math.log(p) for p in precisions) / max_n)`.

</details>

In [ ]:
# @title Fill in solution (Exercise 3)
_fill_in_button([3], "Fill in solution",
                "✓ Now run the exercise cell below: the solution replaces its code.")

In [ ]:
# Exercise 3
import math
from collections import Counter
from sacrebleu.tokenizers.tokenizer_13a import Tokenizer13a

tokenize_13a = Tokenizer13a()


def ngrams(tokens, n):
    # TODO: a Counter of the n-grams in tokens, each n-gram as a tuple of n tokens
    ___


def precision_counts(cand_tokens, ref_tokens, n):
    cand, ref = ngrams(cand_tokens, n), ngrams(ref_tokens, n)
    correct = ___  # TODO: matches, each n-gram clipped to its count in the reference
    total = ___  # TODO: the number of n-grams in the candidate
    return correct, total


def smoothed_precisions(correct, total):
    """Given: sacrebleu's default smoothing. Each zero count gets 1 / (2^k * total)."""
    precisions, factor = [], 1.0
    for c, t in zip(correct, total):
        if c == 0:
            factor *= 2
            precisions.append(1 / (factor * t))
        else:
            precisions.append(c / t)
    return precisions


def my_bleu(candidate, reference, max_n=4):
    cand_tokens = tokenize_13a(candidate).split()
    ref_tokens = tokenize_13a(reference).split()
    counts = [precision_counts(cand_tokens, ref_tokens, n) for n in range(1, max_n + 1)]
    correct = [c for c, _ in counts]
    total = [t for _, t in counts]
    if sum(correct) == 0:
        return 0.0
    c, r = len(cand_tokens), len(ref_tokens)
    bp = ___  # TODO: the brevity penalty
    precisions = smoothed_precisions(correct, total)
    # TODO: return 100 * BP * the geometric mean of the precisions
    ___


for name, text in candidates.items():
    print(f"{name:18s} mine = {my_bleu(text, REFERENCE_EN):6.2f}   sacrebleu = {bleu_scores[name]:6.2f}")

In [ ]:
# Check for Exercise 3. Run this cell as is.
assert ngrams(["a", "b", "a", "b"], 2) == Counter({("a", "b"): 2, ("b", "a"): 1})
assert precision_counts(["the", "the", "the"], ["the", "cat"], 1) == (1, 3), \
    "'the' occurs once in the reference, so it is correct at most once"
for _c, _r in [("the cat sat on a rug", "the cat sat on the mat"), ("on the mat the cat sat", "the cat sat on the mat")]:
    assert abs(my_bleu(_c, _r) - sacrebleu.corpus_bleu([_c], [[_r]]).score) < 1e-6, (_c, my_bleu(_c, _r))
for _name, _text in candidates.items():
    assert abs(my_bleu(_text, REFERENCE_EN) - bleu_scores[_name]) < 1e-6, _name
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
import math
from collections import Counter
from sacrebleu.tokenizers.tokenizer_13a import Tokenizer13a

tokenize_13a = Tokenizer13a()


def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def precision_counts(cand_tokens, ref_tokens, n):
    cand, ref = ngrams(cand_tokens, n), ngrams(ref_tokens, n)
    correct = sum(min(count, ref[g]) for g, count in cand.items())
    total = sum(cand.values())
    return correct, total


def smoothed_precisions(correct, total):
    """Given: sacrebleu's default smoothing. Each zero count gets 1 / (2^k * total)."""
    precisions, factor = [], 1.0
    for c, t in zip(correct, total):
        if c == 0:
            factor *= 2
            precisions.append(1 / (factor * t))
        else:
            precisions.append(c / t)
    return precisions


def my_bleu(candidate, reference, max_n=4):
    cand_tokens = tokenize_13a(candidate).split()
    ref_tokens = tokenize_13a(reference).split()
    counts = [precision_counts(cand_tokens, ref_tokens, n) for n in range(1, max_n + 1)]
    correct = [c for c, _ in counts]
    total = [t for _, t in counts]
    if sum(correct) == 0:
        return 0.0
    c, r = len(cand_tokens), len(ref_tokens)
    bp = 1.0 if c >= r else math.exp(1 - r / c)
    precisions = smoothed_precisions(correct, total)
    return 100 * bp * math.exp(sum(math.log(p) for p in precisions) / max_n)


for name, text in candidates.items():
    print(f"{name:18s} mine = {my_bleu(text, REFERENCE_EN):6.2f}   sacrebleu = {bleu_scores[name]:6.2f}")
```

</details>

In [ ]:
# Stuck on Exercise 3? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 3" above; your own cell stays as it is.)
get_ipython().set_next_input("import math\nfrom collections import Counter\nfrom sacrebleu.tokenizers.tokenizer_13a import Tokenizer13a\n\ntokenize_13a = Tokenizer13a()\n\n\ndef ngrams(tokens, n):\n    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))\n\n\ndef precision_counts(cand_tokens, ref_tokens, n):\n    cand, ref = ngrams(cand_tokens, n), ngrams(ref_tokens, n)\n    correct = sum(min(count, ref[g]) for g, count in cand.items())\n    total = sum(cand.values())\n    return correct, total\n\n\ndef smoothed_precisions(correct, total):\n    \"\"\"Given: sacrebleu's default smoothing. Each zero count gets 1 / (2^k * total).\"\"\"\n    precisions, factor = [], 1.0\n    for c, t in zip(correct, total):\n        if c == 0:\n            factor *= 2\n            precisions.append(1 / (factor * t))\n        else:\n            precisions.append(c / t)\n    return precisions\n\n\ndef my_bleu(candidate, reference, max_n=4):\n    cand_tokens = tokenize_13a(candidate).split()\n    ref_tokens = tokenize_13a(reference).split()\n    counts = [precision_counts(cand_tokens, ref_tokens, n) for n in range(1, max_n + 1)]\n    correct = [c for c, _ in counts]\n    total = [t for _, t in counts]\n    if sum(correct) == 0:\n        return 0.0\n    c, r = len(cand_tokens), len(ref_tokens)\n    bp = 1.0 if c >= r else math.exp(1 - r / c)\n    precisions = smoothed_precisions(correct, total)\n    return 100 * bp * math.exp(sum(math.log(p) for p in precisions) / max_n)\n\n\nfor name, text in candidates.items():\n    print(f\"{name:18s} mine = {my_bleu(text, REFERENCE_EN):6.2f}   sacrebleu = {bleu_scores[name]:6.2f}\")")


### Add the BLEU score to `mt_leaderboard.csv`

Step 8 saved `mt_leaderboard.csv` before this BLEU score existed --- Part 2 runs after
Part 1. This adds it onto the matching rows now, and labels the `general-llm` row
`baseline-fp16`, so Session 2 has something concrete to compare its quantized run
against on both cost and quality.


In [ ]:
mt_leaderboard = pd.read_csv("mt_leaderboard.csv")
mt_leaderboard["bleu_score"] = mt_leaderboard["era"].map(bleu_scores)
mt_leaderboard.loc[mt_leaderboard["era"] == "general-llm", "config"] = "baseline-fp16"
mt_leaderboard.to_csv("mt_leaderboard.csv", index=False)
mt_leaderboard


## Step 11 — BERTScore (2020): a learned, neural metric

BERTScore compares the candidate and the reference using contextual embeddings from a
pretrained language model, matching words by cosine similarity instead of counting exact
overlaps — so it can reward a *correct paraphrase* that BLEU would penalize for not using
the reference's exact words. One real difference from a metric like COMET: BERTScore only
looks at candidate vs. reference, never the source sentence — a genuine methodological
limitation worth remembering when you read the score below, since a fluent-sounding
translation could still be unfaithful to the Galician original and BERTScore alone
wouldn't catch it. We use a smaller model here (`distilbert-base-uncased`) for speed; the
biggest model isn't needed to see the point.

**Known limitation, worth noticing:** this notebook used to use COMET here. Checked
directly against `unbabel-comet`'s own published dependency spec: it pins `numpy<2.0`,
`protobuf<5.0`, and `pytorch-lightning<3.0` — versions old enough now to reliably conflict
with Colab's current base image and crash on import. BERTScore's dependencies are
essentially unpinned, so this should actually run. We keep the same try/except pattern
anyway, since any live network install can still flake mid-class.

In [ ]:
!pip install -q bert-score

bertscore_available = False
bertscore_scores = {}
elapsed_bertscore = 0.0
emissions_bertscore_kg = 0.0

try:
    from bert_score import score as bertscore

    tracker_bertscore = EmissionsTracker(
        project_name="green-eval-bertscore",
        measure_power_secs=1,
        save_to_file=False,
        log_level="warning",
    )
    tracker_bertscore.start()
    start_time = time.time()

    cand_list = list(candidates.values())
    ref_list = [REFERENCE_EN] * len(cand_list)
    _, _, f1 = bertscore(
        cand_list, ref_list, model_type="distilbert-base-uncased", verbose=False
    )
    bertscore_scores = dict(zip(candidates.keys(), f1.tolist()))

    elapsed_bertscore = time.time() - start_time
    emissions_bertscore_kg = tracker_bertscore.stop()
    bertscore_available = True

    for name, s in bertscore_scores.items():
        print(f"{name}: BERTScore F1 = {s:.4f}")
    print(f"\n({elapsed_bertscore:.2f}s, {emissions_bertscore_kg * 1000:.4f} g CO2eq for all candidates)")
except Exception as e:
    print(f"BERTScore failed to load/run in this environment ({type(e).__name__}: {e}).")
    print("Continuing without a BERTScore score — Step 13 will compare BLEU and the LLM")
    print("judge only for this run.")

## Step 12 — LLM-as-a-judge: prompt a model to grade the translation

The newest approach: ask a general-purpose LLM to read the source, the candidate, and the
reference, and output a quality score itself. We reuse the same model from Parts 1 and 2 —
no new download.

In [ ]:
def llm_judge_score(source, candidate, reference):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert translation evaluator. Given a source text, a candidate "
                "translation, and a reference translation, rate the candidate's quality "
                "from 1 (very poor) to 10 (excellent), considering both accuracy and "
                "fluency. Output ONLY the number."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Source (Galician): {source}\n\n"
                f"Reference translation: {reference}\n\n"
                f"Candidate translation: {candidate}\n\n"
                "Score (1-10):"
            ),
        },
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


tracker_judge = EmissionsTracker(
    project_name="green-eval-llm-judge",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_judge.start()
start_time = time.time()

judge_scores = {
    name: llm_judge_score(FULL_TEXT_GL, text, REFERENCE_EN)
    for name, text in candidates.items()
}

elapsed_judge = time.time() - start_time
emissions_judge_kg = tracker_judge.stop()

for name, score in judge_scores.items():
    print(f"{name}: LLM-judge score = {score}")
print(f"\n({elapsed_judge:.2f}s, {emissions_judge_kg * 1000:.4f} g CO2eq for all candidates)")

### A second, independent judge: ChatGPT, by hand

The score above came from the same Qwen model already loaded in this notebook grading
its own family's output --- a real, known weak spot of LLM-as-a-judge: a model can be
biased toward whatever its own family tends to produce. To check that, get a second
opinion from a genuinely different, independent model: ChatGPT (or Claude, Gemini,
whichever you have access to), run by hand, outside this notebook.

The cell below prints one ready-to-paste prompt per candidate --- the exact same prompt
Qwen was just given. Paste each one into your chat tool of choice, one at a time, then
fill the numbers it returns into `CHATGPT_JUDGE_SCORES` in the cell after that.


In [ ]:
def build_judge_prompt(source, candidate, reference):
    return (
        "You are an expert translation evaluator. Given a source text, a candidate "
        "translation, and a reference translation, rate the candidate's quality from 1 "
        "(very poor) to 10 (excellent), considering both accuracy and fluency. Output "
        "ONLY the number.\n\n"
        f"Source (Galician): {source}\n\n"
        f"Reference translation: {reference}\n\n"
        f"Candidate translation: {candidate}\n\n"
        "Score (1-10):"
    )


for name, text in candidates.items():
    print(f"===== prompt for: {name} =====")
    print(build_judge_prompt(FULL_TEXT_GL, text, REFERENCE_EN))
    print()


Replace each `None` below with the number your chat tool returned for that candidate.


In [ ]:
CHATGPT_JUDGE_SCORES = {
    "rule-based": None,
    "statistical": None,
    "neural-dedicated": None,
    "general-llm": None,
}

chatgpt_judge_scores = CHATGPT_JUDGE_SCORES
assert all(v is not None for v in chatgpt_judge_scores.values()), (
    "Fill in every score in CHATGPT_JUDGE_SCORES above before continuing --- some are still None."
)
for name, score in chatgpt_judge_scores.items():
    print(f"{name}: LLM-judge score (ChatGPT, by hand) = {score}")

elapsed_chatgpt_judge = None
emissions_chatgpt_judge_kg = None


## Step 13 — Compare the four evaluation methods

Same four candidate translations, four ways of scoring them — three with a measurable
local cost, one (the ChatGPT judge) that has no local cost by design, since the actual
grading ran outside this notebook.

### What we're actually scoring

The numbers below are only meaningful next to the text they're scoring — here it is
again, right next to the comparison.

In [ ]:
for name, text in candidates.items():
    print(f"{name}:")
    print(f"  {text}")
    print()
print("reference:")
print(f"  {REFERENCE_EN}")

In [ ]:
eval_rows = [
    {"metric": "BLEU (2002)", "needs_gpu": False,
     "duration_sec": round(elapsed_bleu, 4),
     "emissions_g_co2eq": round(emissions_bleu_kg * 1000, 6),
     **{f"{k}_score": round(v, 2) for k, v in bleu_scores.items()}},
]
if bertscore_available:
    eval_rows.append({"metric": "BERTScore (2020)", "needs_gpu": True,
     "duration_sec": round(elapsed_bertscore, 2),
     "emissions_g_co2eq": round(emissions_bertscore_kg * 1000, 4),
     **{f"{k}_score": round(v, 4) for k, v in bertscore_scores.items()}})
eval_rows.append({"metric": "LLM-as-judge (Qwen, today)", "needs_gpu": True,
     "duration_sec": round(elapsed_judge, 2),
     "emissions_g_co2eq": round(emissions_judge_kg * 1000, 4),
     **{f"{k}_score": v for k, v in judge_scores.items()}})
eval_rows.append({"metric": "LLM-as-judge (ChatGPT, by hand)", "needs_gpu": False,
     "duration_sec": elapsed_chatgpt_judge,
     "emissions_g_co2eq": emissions_chatgpt_judge_kg,
     **{f"{k}_score": v for k, v in chatgpt_judge_scores.items()}})

eval_comparison = pd.DataFrame(eval_rows)

print(eval_comparison.to_string(index=False))
if not bertscore_available:
    print("\n(BERTScore is missing from this run — it failed to load in Step 11. This is")
    print("BLEU vs. the LLM judge only.)")
print("\nDo the metrics you have agree on which candidate is better? If not, which would")
print("you trust — and would your answer change if you were scoring 10,000 translations")
print("instead of 2, and had to pay for the energy each method uses?")

### One bar chart per metric

Same idea as the table above, but each metric gets its own panel and its own scale ---
BLEU is only ever compared to BLEU, BERTScore only to BERTScore, and the LLM judge only to
itself, across the four systems. The three metrics are not on comparable scales (BLEU is
roughly 0-100, BERTScore F1 is 0-1, the judge is 1-10), so they are never plotted together
on one axis.


### ✏️ Exercise 4 — Read a number out of a judge's reply

The LLM judge may answer `"8"`, `"Score: 7.5"` or `"7/10"`. Complete `parse_score`
so it returns the first number in the reply as a float, or `None` if there is no number.

<details><summary><b>Hints</b> (click to open)</summary>

- `re.search(pattern, text)` returns the first match, or `None`.
- `r"\d+"` matches one or more digits; `(\.\d+)?` adds an optional decimal part.

</details>

In [ ]:
# @title Fill in solution (Exercise 4)
_fill_in_button([4], "Fill in solution",
                "✓ Now run the exercise cell below: the solution replaces its code.")

In [ ]:
# Exercise 4
import re


def parse_score(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        # TODO: the first number in str(value), as a float, or None
        ___


score_sets = [("BLEU", bleu_scores)]
if bertscore_available:
    score_sets.append(("BERTScore", bertscore_scores))
score_sets.append(("LLM-judge (Qwen)", {k: parse_score(v) for k, v in judge_scores.items()}))
score_sets.append(("LLM-judge (ChatGPT)", {k: parse_score(v) for k, v in chatgpt_judge_scores.items()}))

fig, axes = plt.subplots(1, len(score_sets), figsize=(4.5 * len(score_sets), 4))
if len(score_sets) == 1:
    axes = [axes]

for ax, (metric_name, scores) in zip(axes, score_sets):
    names = list(scores.keys())
    values = [scores[n] for n in names]
    ax.bar(names, values)
    ax.set_title(metric_name)
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
# Check for Exercise 4. Run this cell as is.
assert parse_score("8") == 8.0
assert parse_score("Score: 7.5") == 7.5
assert parse_score("7/10") == 7.0
assert parse_score("no idea") is None
assert parse_score(None) is None
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
import re


def parse_score(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        match = re.search(r"-?\d+(\.\d+)?", str(value))
        return float(match.group()) if match else None


score_sets = [("BLEU", bleu_scores)]
if bertscore_available:
    score_sets.append(("BERTScore", bertscore_scores))
score_sets.append(("LLM-judge (Qwen)", {k: parse_score(v) for k, v in judge_scores.items()}))
score_sets.append(("LLM-judge (ChatGPT)", {k: parse_score(v) for k, v in chatgpt_judge_scores.items()}))

fig, axes = plt.subplots(1, len(score_sets), figsize=(4.5 * len(score_sets), 4))
if len(score_sets) == 1:
    axes = [axes]

for ax, (metric_name, scores) in zip(axes, score_sets):
    names = list(scores.keys())
    values = [scores[n] for n in names]
    ax.bar(names, values)
    ax.set_title(metric_name)
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()
```

</details>

In [ ]:
# Stuck on Exercise 4? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 4" above; your own cell stays as it is.)
get_ipython().set_next_input("import re\n\n\ndef parse_score(value):\n    try:\n        return float(value)\n    except (TypeError, ValueError):\n        match = re.search(r\"-?\\d+(\\.\\d+)?\", str(value))\n        return float(match.group()) if match else None\n\n\nscore_sets = [(\"BLEU\", bleu_scores)]\nif bertscore_available:\n    score_sets.append((\"BERTScore\", bertscore_scores))\nscore_sets.append((\"LLM-judge (Qwen)\", {k: parse_score(v) for k, v in judge_scores.items()}))\nscore_sets.append((\"LLM-judge (ChatGPT)\", {k: parse_score(v) for k, v in chatgpt_judge_scores.items()}))\n\nfig, axes = plt.subplots(1, len(score_sets), figsize=(4.5 * len(score_sets), 4))\nif len(score_sets) == 1:\n    axes = [axes]\n\nfor ax, (metric_name, scores) in zip(axes, score_sets):\n    names = list(scores.keys())\n    values = [scores[n] for n in names]\n    ax.bar(names, values)\n    ax.set_title(metric_name)\n    ax.tick_params(axis=\"x\", rotation=30)\n\nplt.tight_layout()\nplt.show()")


In [ ]:
cheapest_eval = eval_comparison.loc[eval_comparison["emissions_g_co2eq"].idxmin()]
priciest_eval = eval_comparison.loc[eval_comparison["emissions_g_co2eq"].idxmax()]
eval_ratio = priciest_eval["emissions_g_co2eq"] / max(cheapest_eval["emissions_g_co2eq"], 1e-12)

plt.figure(figsize=(7, 4))
plt.bar(eval_comparison["metric"], eval_comparison["emissions_g_co2eq"])
plt.yscale("log")
plt.ylabel("g CO2eq (log scale)")
plt.title(f"Grading cost: ~{eval_ratio:,.0f}x apart, cheapest to priciest")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()
print("(The ChatGPT judge has no bar above --- its cost was never local to this notebook")
print("to measure, not zero.)")

print(f"Cheapest to grade with: {cheapest_eval['metric']} ({cheapest_eval['emissions_g_co2eq']:.6f} g CO2eq).")
print(f"Priciest to grade with: {priciest_eval['metric']} ({priciest_eval['emissions_g_co2eq']:.4f} g CO2eq).")

## Step 14 — Log the evaluation comparison

In [ ]:
# One row per (metric, candidate): duration/energy/emissions were measured once per
# metric, covering every candidate scored in that call together, not one candidate alone
# --- so the cost columns repeat across candidate-rows that share a metric, and only
# "score" varies row to row. This is what lets a later session compare an actual quality
# number against this baseline, not just what grading cost.
leaderboard_rows = []
for name, score in bleu_scores.items():
    leaderboard_rows.append({
        "metric": "bleu", "candidate": name, "score": round(score, 2),
        "duration_sec": elapsed_bleu,
        "energy_kwh": get_tracker_energy_kwh(tracker_bleu), "emissions_kg_co2eq": emissions_bleu_kg,
    })
if bertscore_available:
    for name, score in bertscore_scores.items():
        leaderboard_rows.append({
            "metric": "bertscore", "candidate": name, "score": round(score, 4),
            "duration_sec": elapsed_bertscore,
            "energy_kwh": get_tracker_energy_kwh(tracker_bertscore), "emissions_kg_co2eq": emissions_bertscore_kg,
        })
for name, score in judge_scores.items():
    leaderboard_rows.append({
        "metric": "llm_judge_qwen", "candidate": name, "score": score,
        "duration_sec": elapsed_judge,
        "energy_kwh": get_tracker_energy_kwh(tracker_judge), "emissions_kg_co2eq": emissions_judge_kg,
    })
for name, score in chatgpt_judge_scores.items():
    leaderboard_rows.append({
        "metric": "llm_judge_chatgpt", "candidate": name, "score": score,
        "duration_sec": elapsed_chatgpt_judge,
        "energy_kwh": None, "emissions_kg_co2eq": emissions_chatgpt_judge_kg,
    })

eval_leaderboard = pd.DataFrame(leaderboard_rows)
eval_leaderboard.to_csv("mt_eval_leaderboard.csv", index=False)
eval_leaderboard

In [ ]:
import os

if os.path.exists("mt_eval_leaderboard.csv"):
    print("Saved mt_eval_leaderboard.csv in this Colab session.")
    print("To download it: click the folder icon in the left sidebar, find")
    print("mt_eval_leaderboard.csv, right-click it, and choose Download.")
else:
    print("mt_eval_leaderboard.csv not found — re-run the cell above first.")

## Wrap-up: everything we measured today

Same units (g CO2eq), every technique from this notebook, on one chart.

In [ ]:
summary_rows = [
    {"item": "Part 1: rule-based (Apertium)", "g_co2eq": emissions_rule_kg * 1000},
    {"item": "Part 1: statistical (IBM Model 1)", "g_co2eq": emissions_stat_kg * 1000},
    {"item": "Part 1: neural, dedicated", "g_co2eq": emissions_neural_kg * 1000},
    {"item": "Part 1: general-purpose LLM", "g_co2eq": emissions_llm_kg * 1000},
    {"item": "Part 2: BLEU", "g_co2eq": emissions_bleu_kg * 1000},
]
if bertscore_available:
    summary_rows.append({"item": "Part 2: BERTScore", "g_co2eq": emissions_bertscore_kg * 1000})
summary_rows.append({"item": "Part 2: LLM-as-judge", "g_co2eq": emissions_judge_kg * 1000})

summary = pd.DataFrame(summary_rows)
overall_cheapest = summary.loc[summary["g_co2eq"].idxmin()]
overall_priciest = summary.loc[summary["g_co2eq"].idxmax()]
overall_ratio = overall_priciest["g_co2eq"] / max(overall_cheapest["g_co2eq"], 1e-12)

plt.figure(figsize=(9, 5))
plt.barh(summary["item"], summary["g_co2eq"])
plt.xscale("log")
plt.xlabel("g CO2eq (log scale)")
plt.title(f"Cheapest to priciest technique today: ~{overall_ratio:,.0f}x apart")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(summary.to_string(index=False))
print(f"\nCheapest: {overall_cheapest['item']}. Priciest: {overall_priciest['item']}.")

**Key takeaways**

- Cost spans several orders of magnitude in this notebook, driven by *how* a system is
  built — fixed rules, counting word pairs, a small trained network, or a giant
  general-purpose model each cost fundamentally different amounts to run, independent of
  output quality.
- Model size and cost move independently of each other — now with numbers you measured
  yourself.
- Evaluation has its own cost curve, separate from translation's: BLEU is nearly free, a
  learned neural metric costs real energy, and an LLM-as-judge sits somewhere in between
  depending on the model. Grading carries a real cost too, on top of translating.
- Even "free" or "off-the-shelf" tools carry real friction: Apertium's real compute stays
  invisible to a local CodeCarbon tracker, COMET's pinned dependencies reliably break on
  current Colab images, and CodeCarbon itself falls back to rougher estimates when it
  can't read your GPU or your location. Measuring your own numbers, and understanding
  where those numbers come from, is the actual skill this notebook practices.

## Step 15 — Share-out (discussion)

Before we wrap up, compare notes with your neighbor.

**On the translation comparison (Part 1):**

- Look at the cost table from Step 7. How many orders of magnitude separate the cheapest
  and most expensive approach?
- Neither the real rule-based system nor the toy statistical system could ever produce
  "I had an aunt who..." — why not, structurally? What would it actually take to fix
  that, beyond a bigger dictionary or a better language model?
- Apertium is a real, currently maintained system — did that change how seriously you
  took its limitations, compared to the rule-based approach you might have pictured
  before running this?
- Was the dedicated neural model's translation actually worse than the general-purpose
  LLM's, or just different?
- This notebook's general-purpose LLM is small (1.5B parameters), to keep things running
  on a free Colab GPU. Paste the Galician sentence into a large commercial model (ChatGPT,
  Claude, Gemini). Which opening does it choose: "my aunt", "an aunt of mine", or "I had
  an aunt who..."? Does a larger model restructure the sentence where ours did not?

**On the evaluation comparison (Part 2):**

- Did BLEU, BERTScore, and the LLM judge agree on which translation was better?
- Which one would you trust for a single sentence? For 10,000 sentences, where the cost of
  scoring itself starts to matter?

## Step 16 — Code literacy check

Before you're done, pick at least two or three code cells above that you actually ran and
read today — across any of Parts 1-3 — and add `#` comments to them explaining, in your
own words, what each meaningful line or block does and *why* it's there, in language a
classmate who hasn't seen the code could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?

This is a self-check on your own understanding: a way to catch, in your own words, where
your mental model of the code breaks down against a second reader.